In [1]:
import pandas as pd
import numpy as np
import pencilbox as pb
import datetime as dt
import time
from datetime import datetime, timedelta

In [2]:
conn = pb.get_connection("[Warehouse] Trino")
pd.set_option('display.max_columns', None)  

In [3]:
def read_sql(sql, con = conn):
    max_tries = 3
    for attempt in range(max_tries):
        print(f"Read attempt: {attempt}...")
        try:
            start = time.time()
            df = pd.read_sql_query(sql, con)
            end = time.time()
            if (end - start) > 60:
                print("Time: ", (end - start) / 60, "min")
            else:
                print("Time: ", end - start, "s")
            return df
            break
        except BaseException as e:
            print(e)
            time.sleep(5)

In [4]:
fixed_ppi=f"""

 WITH fixed_hourly_orders AS (
    SELECT
        f.outlet_id,
        f.order_checkout_dt_ist AS date_,
        DATE_TRUNC('hour', f.order_checkout_ts_ist) AS hour_ts,
        DATE_DIFF('second', f.order_picking_started_ts_ist, f.order_picking_completed_ts_ist) AS picking_time,
        f.total_items_quantity_delivered
    FROM dwh.fact_supply_chain_order_details f
    JOIN ops_management.user u
        ON f.picker_id = u.employee_id
    WHERE 
        u.designation != 'Captain OD'
        --AND f.outlet_id = 1024
        AND f.order_checkout_dt_ist  >= CURRENT_DATE-interval '1' day AND  f.order_checkout_dt_ist <= CURRENT_DATE
        AND f.is_rescheduled=false 
        AND f.order_picking_completed_ts_ist is not null 
        AND f.order_type in ('RetailForwardOrder','DropShippingForwardOrder')
),
hourly_with_slots AS (
    SELECT *,
        HOUR(hour_ts) AS hour_val,
        CASE 
            WHEN HOUR(hour_ts) IN (6,7,8,9,10,11) THEN 'Morning'
            WHEN HOUR(hour_ts) IN (12,13,14,15,16,17) THEN 'Afternoon'
            WHEN HOUR(hour_ts) IN (18,19,20,21,22) THEN 'Evening'
            ELSE NULL
        END AS slot_bucket
    FROM fixed_hourly_orders
),
fixed_ppi_data AS (
    SELECT
        date_,
        outlet_id,
        slot_bucket,
        SUM(picking_time) AS total_busy_time,
        SUM(total_items_quantity_delivered) AS total_items_quantity_delivered,
        CASE 
            WHEN SUM(total_items_quantity_delivered) = 0 THEN 0
            ELSE SUM(picking_time) / SUM(total_items_quantity_delivered)
        END AS fixed_busy_ppi
    FROM hourly_with_slots
    WHERE slot_bucket IS NOT NULL
    GROUP BY date_, outlet_id, slot_bucket
),
median_fixed_ppi AS (
    SELECT
        outlet_id,
        slot_bucket,
        approx_percentile(fixed_busy_ppi, 0.5) AS median_fixed_busy_ppi
    FROM fixed_ppi_data
    GROUP BY outlet_id, slot_bucket
)
SELECT
    outlet_id,
    slot_bucket,
    median_fixed_busy_ppi AS projected_today_fixed_ppi
FROM median_fixed_ppi
ORDER BY outlet_id, slot_bucket
"""
fixed_ppi=read_sql(fixed_ppi,conn)
fixed_ppi.head()

Read attempt: 0...
Time:  17.73287606239319 s


,outlet_id,slot_bucket,projected_today_fixed_ppi
0,1024,Afternoon,21.855470
1,1024,Evening,21.159872
2,1024,Morning,19.538298
3,1122,Afternoon,17.008762
4,1122,Evening,16.983570


In [5]:
OD_ppi = f"""

with od_hourly_orders AS (
    SELECT
        f.outlet_id,
        f.order_checkout_dt_ist AS date_,
        DATE_TRUNC('hour', f.order_checkout_ts_ist) AS hour_ts,
        DATE_DIFF('second', f.order_picking_started_ts_ist, f.order_picking_completed_ts_ist) AS picking_time,
        f.total_items_quantity_delivered
    FROM dwh.fact_supply_chain_order_details f
    JOIN ops_management.user u
        ON f.picker_id = u.employee_id
    WHERE 
        u.designation = 'Captain OD' 
        AND f.order_checkout_dt_ist  >= CURRENT_DATE-interval '1' day AND  f.order_checkout_dt_ist <= CURRENT_DATE 
        AND f.is_rescheduled=false 
        --AND outlet_id=1024
        AND f.order_picking_completed_ts_ist is not null 
        AND f.order_type in ('RetailForwardOrder','DropShippingForwardOrder')
),

shourly_with_slots AS (
    SELECT *,
        HOUR(hour_ts) AS hour_val,
        CASE 
            WHEN HOUR(hour_ts) IN (6,7,8,9,10,11) THEN 'Morning'
            WHEN HOUR(hour_ts) IN (12,13,14,15,16,17) THEN 'Afternoon'
            WHEN HOUR(hour_ts) IN (18,19,20,21,22) THEN 'Evening'
            ELSE NULL
        END AS slot_bucket
    FROM od_hourly_orders
),
od_ppi_data AS (
    SELECT
        date_,
        outlet_id,
        slot_bucket,
        SUM(picking_time) AS total_busy_time,
        SUM(total_items_quantity_delivered) AS total_items_quantity_delivered,
        CASE 
            WHEN SUM(total_items_quantity_delivered) = 0 THEN 0
            ELSE SUM(picking_time) / SUM(total_items_quantity_delivered)
        END AS od_busy_ppi
    FROM shourly_with_slots
    WHERE slot_bucket IS NOT NULL
    GROUP BY date_, outlet_id, slot_bucket
),
median_od_ppi AS (
    SELECT
        outlet_id,
        slot_bucket,
        approx_percentile(od_busy_ppi, 0.5) AS median_OD_busy_ppi
    FROM od_ppi_data
    GROUP BY outlet_id, slot_bucket
)
SELECT
    outlet_id,
    slot_bucket,
    median_OD_busy_ppi AS projected_today_od_ppi
FROM median_od_ppi
ORDER BY outlet_id, slot_bucket
"""
OD_ppi=read_sql(OD_ppi,conn)
OD_ppi.head()

Read attempt: 0...
Time:  15.311776161193848 s


,outlet_id,slot_bucket,projected_today_od_ppi
0,1024,Afternoon,18.506369
1,1024,Evening,20.010370
2,1024,Morning,13.398371
3,1122,Afternoon,18.650442
4,1122,Evening,16.133240


In [6]:
df = pd.read_csv("/shared/Shared/Yogesh Jindal/Store Milestone/milestone_store_tag.csv")
df.rename(columns={'Outlet_id': 'outlet_id'}, inplace=True)
df.fillna(0,inplace=True)
df.head()

,outlet_id,Double Story OWN,Lt_stores,split_pick
0,1024,0.0,0.0,0.0
1,1122,0.0,0.0,0.0
2,1314,0.0,0.0,0.0
3,1337,0.0,0.0,0.0
4,1387,0.0,0.0,0.0


In [7]:
merged1=pd.merge(
    fixed_ppi,OD_ppi,
    on=["outlet_id", "slot_bucket"],
    how="left"
)
merged1=pd.merge(
    merged1,df,
    on=["outlet_id"],
    how="left"
)
merged1 = merged1[['outlet_id','slot_bucket','projected_today_fixed_ppi','projected_today_od_ppi','Double Story OWN','Lt_stores','split_pick']]
# merged1[merged1["outlet_id"] == 4414]
merged1.head()

,outlet_id,slot_bucket,projected_today_fixed_ppi,projected_today_od_ppi,Double Story OWN,Lt_stores,split_pick
0,1024,Afternoon,21.855470,18.506369,0.0,0.0,0.0
1,1024,Evening,21.159872,20.010370,0.0,0.0,0.0
2,1024,Morning,19.538298,13.398371,0.0,0.0,0.0
3,1122,Afternoon,17.008762,18.650442,0.0,0.0,0.0
4,1122,Evening,16.983570,16.133240,0.0,0.0,0.0


In [8]:
merged1['util_double'] = np.where(merged1['Double Story OWN'].fillna(0) == 1, 0.60, np.nan)
merged1['util_lt']     = np.where(merged1['Lt_stores'].fillna(0) == 1, 0.55, np.nan)
merged1['util_split']  = np.where(merged1['split_pick'].fillna(0) == 1, 0.55, np.nan)

merged1['median_util'] = merged1[['util_double','util_lt','util_split']].min(axis=1)

merged1['median_util'] = merged1['median_util'].fillna(0.65)

In [9]:
# merged1['median_util'] = np.where(
#         merged1['Double Story OWN'].fillna(0) ==1,
#     0.60,
#     0.65
# )

merged1['two_min_ppi'] = np.where(
    merged1['projected_today_od_ppi'].fillna(0)==0,
    merged1['projected_today_fixed_ppi'],
    np.minimum(merged1['projected_today_fixed_ppi'],merged1['projected_today_od_ppi'])
)
    
merged1["two_min_ppi"] = merged1["two_min_ppi"].clip(10,30)    
merged1["two_min_ppi"] = merged1["two_min_ppi"].round(2)
# merged1[merged1["outlet_id"] == 7005]
merged1.head()

,outlet_id,slot_bucket,projected_today_fixed_ppi,projected_today_od_ppi,Double Story OWN,Lt_stores,split_pick,util_double,util_lt,util_split,median_util,two_min_ppi
0,1024,Afternoon,21.855470,18.506369,0.0,0.0,0.0,NaN,NaN,NaN,0.65,18.51
1,1024,Evening,21.159872,20.010370,0.0,0.0,0.0,NaN,NaN,NaN,0.65,20.01
2,1024,Morning,19.538298,13.398371,0.0,0.0,0.0,NaN,NaN,NaN,0.65,13.40
3,1122,Afternoon,17.008762,18.650442,0.0,0.0,0.0,NaN,NaN,NaN,0.65,17.01
4,1122,Evening,16.983570,16.133240,0.0,0.0,0.0,NaN,NaN,NaN,0.65,16.13


### Till Here based on 2 days data

In [10]:
fif_day_od = f"""
with od_hourly_orders AS (
    SELECT
        f.outlet_id,
        f.order_checkout_dt_ist AS date_,
        DATE_TRUNC('hour', f.order_checkout_ts_ist) AS hour_ts,
        DATE_DIFF('second', f.order_picking_started_ts_ist, f.order_picking_completed_ts_ist) AS picking_time,
        f.total_items_quantity_delivered
    FROM dwh.fact_supply_chain_order_details f
    JOIN ops_management.user u
        ON f.picker_id = u.employee_id
    WHERE 
        u.designation = 'Captain OD' 
        AND f.order_checkout_dt_ist  >= current_date - interval '14' day AND  f.order_checkout_dt_ist <= CURRENT_DATE 
        AND f.is_rescheduled=false 
        --AND outlet_id=1024
        AND f.order_picking_completed_ts_ist is not null 
        AND f.order_type in ('RetailForwardOrder','DropShippingForwardOrder')
),

shourly_with_slots AS (
    SELECT *,
        HOUR(hour_ts) AS hour_val,
        CASE 
            WHEN HOUR(hour_ts) IN (6,7,8,9,10,11) THEN 'Morning'
            WHEN HOUR(hour_ts) IN (12,13,14,15,16,17) THEN 'Afternoon'
            WHEN HOUR(hour_ts) IN (18,19,20,21,22) THEN 'Evening'
            ELSE NULL
        END AS slot_bucket
    FROM od_hourly_orders
),
od_ppi_data AS (
    SELECT
        date_,
        outlet_id,
        slot_bucket,
        SUM(picking_time) AS total_busy_time,
        SUM(total_items_quantity_delivered) AS total_items_quantity_delivered,
        CASE 
            WHEN SUM(total_items_quantity_delivered) = 0 THEN 0
            ELSE SUM(picking_time) / SUM(total_items_quantity_delivered)
        END AS od_busy_ppi
    FROM shourly_with_slots
    WHERE slot_bucket IS NOT NULL
    GROUP BY date_, outlet_id, slot_bucket
),
median_od_ppi AS (
    SELECT
        outlet_id,
        slot_bucket,
        approx_percentile(od_busy_ppi, 0.5) AS median_OD_busy_ppi
    FROM od_ppi_data
    GROUP BY outlet_id, slot_bucket
)
SELECT
    outlet_id,
    slot_bucket,
    median_OD_busy_ppi AS fif_day_od_ppi
FROM median_od_ppi
ORDER BY outlet_id, slot_bucket
"""
fif_day_od=read_sql(fif_day_od,conn)
fif_day_od.head()

Read attempt: 0...
Time:  20.84791374206543 s


,outlet_id,slot_bucket,fif_day_od_ppi
0,1024,Afternoon,18.982286
1,1024,Evening,18.589659
2,1024,Morning,13.957275
3,1122,Afternoon,18.074350
4,1122,Evening,16.686762


In [11]:
fif_day_ft = f"""
WITH fixed_hourly_orders AS (
    SELECT
        f.outlet_id,
        f.order_checkout_dt_ist AS date_,
        DATE_TRUNC('hour', f.order_checkout_ts_ist) AS hour_ts,
        DATE_DIFF('second', f.order_picking_started_ts_ist, f.order_picking_completed_ts_ist) AS picking_time,
        f.total_items_quantity_delivered
    FROM dwh.fact_supply_chain_order_details f
    JOIN ops_management.user u
        ON f.picker_id = u.employee_id
    WHERE 
        u.designation != 'Captain OD'
        --AND f.outlet_id = 1024
        AND f.order_checkout_dt_ist >= current_date - interval '14' day  AND  f.order_checkout_dt_ist <= CURRENT_DATE
        AND f.is_rescheduled=false
        AND f.order_picking_completed_ts_ist is not null 
        AND f.order_type in ('RetailForwardOrder','DropShippingForwardOrder')
),
hourly_with_slots AS (
    SELECT *,
        HOUR(hour_ts) AS hour_val,
        CASE 
            WHEN HOUR(hour_ts) IN (6,7,8,9,10,11) THEN 'Morning'
            WHEN HOUR(hour_ts) IN (12,13,14,15,16,17) THEN 'Afternoon'
            WHEN HOUR(hour_ts) IN (18,19,20,21,22) THEN 'Evening'
            ELSE NULL
        END AS slot_bucket
    FROM fixed_hourly_orders
),
fixed_ppi_data AS (
    SELECT
        date_,
        outlet_id,
        slot_bucket,
        SUM(picking_time) AS total_busy_time,
        SUM(total_items_quantity_delivered) AS total_items_quantity_delivered,
        CASE 
            WHEN SUM(total_items_quantity_delivered) = 0 THEN 0
            ELSE SUM(picking_time) / SUM(total_items_quantity_delivered)
        END AS fixed_busy_ppi
    FROM hourly_with_slots
    WHERE slot_bucket IS NOT NULL
    GROUP BY date_, outlet_id, slot_bucket
),
median_fixed_ppi AS (
    SELECT
        outlet_id,
        slot_bucket,
        approx_percentile(fixed_busy_ppi, 0.5) AS median_fixed_busy_ppi
    FROM fixed_ppi_data
    GROUP BY outlet_id, slot_bucket
)
SELECT
    outlet_id,
    slot_bucket,
    median_fixed_busy_ppi AS fif_day_ft_ppi
FROM median_fixed_ppi
ORDER BY outlet_id, slot_bucket
"""
fif_day_ft=read_sql(fif_day_ft,conn)
fif_day_ft.head()

Read attempt: 0...
Time:  19.133193254470825 s


,outlet_id,slot_bucket,fif_day_ft_ppi
0,1024,Afternoon,18.721285
1,1024,Evening,20.811357
2,1024,Morning,14.356835
3,1122,Afternoon,18.816990
4,1122,Evening,20.094510


In [12]:
merged1=pd.merge(
    merged1,fif_day_ft,
    on=["outlet_id", "slot_bucket"],
    how="left"
)
merged1=pd.merge(
    merged1,fif_day_od,
    on=["outlet_id", "slot_bucket"],
    how="left"
)
# merged1[merged1["outlet_id"] == 4414]
merged1.head()

,outlet_id,slot_bucket,projected_today_fixed_ppi,projected_today_od_ppi,Double Story OWN,Lt_stores,split_pick,util_double,util_lt,util_split,median_util,two_min_ppi,fif_day_ft_ppi,fif_day_od_ppi
0,1024,Afternoon,21.855470,18.506369,0.0,0.0,0.0,NaN,NaN,NaN,0.65,18.51,18.721285,18.982286
1,1024,Evening,21.159872,20.010370,0.0,0.0,0.0,NaN,NaN,NaN,0.65,20.01,20.811357,18.589659
2,1024,Morning,19.538298,13.398371,0.0,0.0,0.0,NaN,NaN,NaN,0.65,13.40,14.356835,13.957275
3,1122,Afternoon,17.008762,18.650442,0.0,0.0,0.0,NaN,NaN,NaN,0.65,17.01,18.816990,18.074350
4,1122,Evening,16.983570,16.133240,0.0,0.0,0.0,NaN,NaN,NaN,0.65,16.13,20.094510,16.686762


In [13]:
merged1['fif_min_ppi'] = np.where(
    merged1['fif_day_od_ppi'].fillna(0)==0,
    merged1['fif_day_ft_ppi'],
    np.minimum(merged1['fif_day_ft_ppi'],merged1['fif_day_od_ppi'])
)
merged1["fif_min_ppi"] = merged1["fif_min_ppi"].clip(10,30) 
merged1["fif_min_ppi"] = merged1["fif_min_ppi"].round(2)
# merged1[merged1["outlet_id"] == 7005]
merged1.head()

,outlet_id,slot_bucket,projected_today_fixed_ppi,projected_today_od_ppi,Double Story OWN,Lt_stores,split_pick,util_double,util_lt,util_split,median_util,two_min_ppi,fif_day_ft_ppi,fif_day_od_ppi,fif_min_ppi
0,1024,Afternoon,21.855470,18.506369,0.0,0.0,0.0,NaN,NaN,NaN,0.65,18.51,18.721285,18.982286,18.72
1,1024,Evening,21.159872,20.010370,0.0,0.0,0.0,NaN,NaN,NaN,0.65,20.01,20.811357,18.589659,18.59
2,1024,Morning,19.538298,13.398371,0.0,0.0,0.0,NaN,NaN,NaN,0.65,13.40,14.356835,13.957275,13.96
3,1122,Afternoon,17.008762,18.650442,0.0,0.0,0.0,NaN,NaN,NaN,0.65,17.01,18.816990,18.074350,18.07
4,1122,Evening,16.983570,16.133240,0.0,0.0,0.0,NaN,NaN,NaN,0.65,16.13,20.094510,16.686762,16.69


### Till Here combined

In [14]:
# merged2['plan_ppi'] = np.where(
#         (merged2["two_min_ppi"] < merged2["fif_min_ppi"]) | (merged2["two_min_ppi"] > merged2["fif_min_ppi"]+merged2['delta_ppi']),
#         np.minimum(np.maximum(merged2["two_min_ppi"], merged2["fif_min_ppi"]-merged2['delta_ppi']), merged2["fif_min_ppi"]+merged2['delta_ppi']),
#         merged2["two_min_ppi"]
#     )


merged1['ppi_final'] = np.where(
        (merged1["two_min_ppi"].fillna(0) > merged1["fif_min_ppi"].fillna(0)),
        np.minimum(merged1["two_min_ppi"],merged1["fif_min_ppi"]+1),
        merged1["two_min_ppi"]
    )


# merged1['plan_ppi'] = np.where(
#         (merged1["two_min_ppi"] > merged1["fif_min_ppi"]),
#         merged1["fif_min_ppi"],
#         np.maximum(merged1["two_min_ppi"], merged1["fif_min_ppi"]-2)
#     )
# merged1[merged1["outlet_id"] == 7005]
merged1.head()      

,outlet_id,slot_bucket,projected_today_fixed_ppi,projected_today_od_ppi,Double Story OWN,Lt_stores,split_pick,util_double,util_lt,util_split,median_util,two_min_ppi,fif_day_ft_ppi,fif_day_od_ppi,fif_min_ppi,ppi_final
0,1024,Afternoon,21.855470,18.506369,0.0,0.0,0.0,NaN,NaN,NaN,0.65,18.51,18.721285,18.982286,18.72,18.51
1,1024,Evening,21.159872,20.010370,0.0,0.0,0.0,NaN,NaN,NaN,0.65,20.01,20.811357,18.589659,18.59,19.59
2,1024,Morning,19.538298,13.398371,0.0,0.0,0.0,NaN,NaN,NaN,0.65,13.40,14.356835,13.957275,13.96,13.40
3,1122,Afternoon,17.008762,18.650442,0.0,0.0,0.0,NaN,NaN,NaN,0.65,17.01,18.816990,18.074350,18.07,17.01
4,1122,Evening,16.983570,16.133240,0.0,0.0,0.0,NaN,NaN,NaN,0.65,16.13,20.094510,16.686762,16.69,16.13


In [15]:
merged1['M1'] = (7200 * merged1['median_util']) / (merged1['ppi_final']*1.2)
merged1['M2'] = (7200 * merged1['median_util']) / merged1['ppi_final']
merged1['M3'] = (7200 * merged1['median_util']) / (merged1['ppi_final']*0.75)
# merged1[merged1["outlet_id"] == 4414]
merged1.head()

,outlet_id,slot_bucket,projected_today_fixed_ppi,projected_today_od_ppi,Double Story OWN,Lt_stores,split_pick,util_double,util_lt,util_split,median_util,two_min_ppi,fif_day_ft_ppi,fif_day_od_ppi,fif_min_ppi,ppi_final,M1,M2,M3
0,1024,Afternoon,21.855470,18.506369,0.0,0.0,0.0,NaN,NaN,NaN,0.65,18.51,18.721285,18.982286,18.72,18.51,210.696921,252.836305,337.115073
1,1024,Evening,21.159872,20.010370,0.0,0.0,0.0,NaN,NaN,NaN,0.65,20.01,20.811357,18.589659,18.59,19.59,199.081164,238.897397,318.529862
2,1024,Morning,19.538298,13.398371,0.0,0.0,0.0,NaN,NaN,NaN,0.65,13.40,14.356835,13.957275,13.96,13.40,291.044776,349.253731,465.671642
3,1122,Afternoon,17.008762,18.650442,0.0,0.0,0.0,NaN,NaN,NaN,0.65,17.01,18.816990,18.074350,18.07,17.01,229.276896,275.132275,366.843034
4,1122,Evening,16.983570,16.133240,0.0,0.0,0.0,NaN,NaN,NaN,0.65,16.13,20.094510,16.686762,16.69,16.13,241.785493,290.142591,386.856789


In [16]:
merged1["M1"] = merged1["M1"].round(0)
merged1["M2"] = merged1["M2"].round(0)
merged1["M3"] = merged1["M3"].round(0)
# merged1[merged1["outlet_id"] == 7005]
merged1.head(10)

,outlet_id,slot_bucket,projected_today_fixed_ppi,projected_today_od_ppi,Double Story OWN,Lt_stores,split_pick,util_double,util_lt,util_split,median_util,two_min_ppi,fif_day_ft_ppi,fif_day_od_ppi,fif_min_ppi,ppi_final,M1,M2,M3
0,1024,Afternoon,21.855470,18.506369,0.0,0.0,0.0,NaN,NaN,NaN,0.65,18.51,18.721285,18.982286,18.72,18.51,211.0,253.0,337.0
1,1024,Evening,21.159872,20.010370,0.0,0.0,0.0,NaN,NaN,NaN,0.65,20.01,20.811357,18.589659,18.59,19.59,199.0,239.0,319.0
2,1024,Morning,19.538298,13.398371,0.0,0.0,0.0,NaN,NaN,NaN,0.65,13.40,14.356835,13.957275,13.96,13.40,291.0,349.0,466.0
3,1122,Afternoon,17.008762,18.650442,0.0,0.0,0.0,NaN,NaN,NaN,0.65,17.01,18.816990,18.074350,18.07,17.01,229.0,275.0,367.0
4,1122,Evening,16.983570,16.133240,0.0,0.0,0.0,NaN,NaN,NaN,0.65,16.13,20.094510,16.686762,16.69,16.13,242.0,290.0,387.0
5,1122,Morning,16.041860,16.617977,0.0,0.0,0.0,NaN,NaN,NaN,0.65,16.04,19.123823,18.469309,18.47,16.04,243.0,292.0,389.0
6,1314,Afternoon,26.118011,24.372760,0.0,0.0,0.0,NaN,NaN,NaN,0.65,24.37,34.721615,28.907047,28.91,24.37,160.0,192.0,256.0
7,1314,Evening,37.102203,21.784742,0.0,0.0,0.0,NaN,NaN,NaN,0.65,21.78,36.823017,27.424192,27.42,21.78,179.0,215.0,287.0
8,1314,Morning,17.820564,19.424006,0.0,0.0,0.0,NaN,NaN,NaN,0.65,17.82,29.413174,22.434700,22.43,17.82,219.0,263.0,350.0
9,1337,Afternoon,33.220722,21.147360,0.0,0.0,0.0,NaN,NaN,NaN,0.65,21.15,29.657267,19.323128,19.32,20.32,192.0,230.0,307.0


In [17]:
# merged1.to_csv("fif_raw1.csv",index=False)

In [18]:

prev_mile = f"""
with prev_mile as(
    select 
    id,
    site_id as outlet_id, 
    date(start_time + interval '330' minute) as date_ts,
    start_time + interval '330' minute as start_time,
    end_time + interval '330' minute as end_time,
    cast(json_extract(meta, '$.payout.minimum_guarantee') as double) as minimum_guarantee,
    cast(json_extract_scalar(json_extract(meta, '$.milestones[0]'), '$.item_count') as double) as item_count_1,
    cast(json_extract_scalar(json_extract(meta, '$.milestones[0]'), '$.bonus_rupees') as double) as bonus_rupees_1,
    cast(json_extract_scalar(json_extract(meta, '$.milestones[1]'), '$.item_count') as double) as item_count_2,
    cast(json_extract_scalar(json_extract(meta, '$.milestones[1]'), '$.bonus_rupees') as double) as bonus_rupees_2,
    cast(json_extract_scalar(json_extract(meta, '$.milestones[2]'), '$.item_count') as double) as item_count_3,
    cast(json_extract_scalar(json_extract(meta, '$.milestones[2]'), '$.bonus_rupees') as double) as bonus_rupees_3,
    cast(json_extract(meta, '$.payout.penalty_rupees') as double) as penalty_rupees,
    cast(json_extract_scalar(meta, '$.payout.cost_per_item_picked') as double) as cost_per_item_picked,
    cast(json_extract(meta, '$.bonus.slot_bonus_rupees') as double) as slot_bonus_rupees,
    row_number() over(partition by id order by update_ts desc) as rnk
    from ops_management.slots
    --where insert_ds_ist BETWEEN CAST(CURRENT_DATE -interval '4' day AS VARCHAR) AND CAST(CURRENT_DATE AS VARCHAR)
    where insert_ds_ist = CAST(CURRENT_DATE AS VARCHAR)
    and slot_type = 'ON_DEMAND_PICKING'
    AND status = 'ACTIVE'
    --AND site_id=CAST(1122 as varchar)
    ORDER BY insert_ds_ist DESC
),
pslot_bucket AS (
  SELECT *,
    CASE 
      WHEN format_datetime(start_time, 'HH:mm') IN ('08:00', '09:00', '10:00','11:00','12:00') THEN 'Morning'
      WHEN format_datetime(start_time, 'HH:mm') IN ('13:00', '14:00', '15:00','16:00','17:00') THEN 'Afternoon'
      WHEN format_datetime(start_time, 'HH:mm') IN ('18:00', '19:00', '20:00', '21:00','22:00') THEN 'Evening'
      ELSE 'Other'
    END AS slot_bucket
  FROM prev_mile
)

  SELECT
    date_ts,
    outlet_id,
    slot_bucket,
    approx_percentile(item_count_1, 0.5) AS prev_M1,
    approx_percentile(item_count_2, 0.5) AS prev_M2,
    approx_percentile(item_count_3, 0.5) AS prev_M3
  FROM pslot_bucket
  WHERE slot_bucket != 'Other' 
  GROUP BY date_ts,outlet_id, slot_bucket
  """
prev_mile = read_sql(prev_mile, conn)
prev_mile.head()

Read attempt: 0...
Time:  7.5241615772247314 s


,date_ts,outlet_id,slot_bucket,prev_M1,prev_M2,prev_M3
0,2026-06-04,7689,Evening,90.0,110.0,145.0
1,2026-06-04,5060,Morning,245.0,295.0,395.0
2,2026-06-04,7346,Evening,215.0,255.0,340.0
3,2026-06-04,7930,Evening,225.0,270.0,360.0
4,2026-06-04,6554,Afternoon,195.0,235.0,310.0


In [19]:
merged1['outlet_id'] = merged1['outlet_id'].astype(str)
merged2=pd.merge(
    merged1,prev_mile,
    on=["outlet_id", "slot_bucket"],
    how="left"
)  
# merged2[merged2["outlet_id"] == 7181]
merged2.head()

,outlet_id,slot_bucket,projected_today_fixed_ppi,projected_today_od_ppi,Double Story OWN,Lt_stores,split_pick,util_double,util_lt,util_split,median_util,two_min_ppi,fif_day_ft_ppi,fif_day_od_ppi,fif_min_ppi,ppi_final,M1,M2,M3,date_ts,prev_M1,prev_M2,prev_M3
0,1024,Afternoon,21.855470,18.506369,0.0,0.0,0.0,NaN,NaN,NaN,0.65,18.51,18.721285,18.982286,18.72,18.51,211.0,253.0,337.0,2026-06-04,230.0,275.0,365.0
1,1024,Evening,21.159872,20.010370,0.0,0.0,0.0,NaN,NaN,NaN,0.65,20.01,20.811357,18.589659,18.59,19.59,199.0,239.0,319.0,2026-06-04,230.0,275.0,365.0
2,1024,Morning,19.538298,13.398371,0.0,0.0,0.0,NaN,NaN,NaN,0.65,13.40,14.356835,13.957275,13.96,13.40,291.0,349.0,466.0,2026-06-04,290.0,350.0,465.0
3,1122,Afternoon,17.008762,18.650442,0.0,0.0,0.0,NaN,NaN,NaN,0.65,17.01,18.816990,18.074350,18.07,17.01,229.0,275.0,367.0,2026-06-04,210.0,250.0,335.0
4,1122,Evening,16.983570,16.133240,0.0,0.0,0.0,NaN,NaN,NaN,0.65,16.13,20.094510,16.686762,16.69,16.13,242.0,290.0,387.0,2026-06-04,215.0,255.0,340.0


In [20]:

merged2['M1_amount'] = np.where(
    merged2['Double Story OWN'].fillna(0) == 1,
    merged2['slot_bucket'].map({
        'Morning': 35,
        'Afternoon': 30,
        'Evening': 35
    }),
    merged2['slot_bucket'].map({
        'Morning': 25,
        'Afternoon': 20,
        'Evening': 25
    })
)

merged2['M2_amount'] = np.where(
    merged2['Double Story OWN'].fillna(0) == 1,
    merged2['slot_bucket'].map({
        'Morning': 60,
        'Afternoon': 50,
        'Evening': 60
    }),
    merged2['slot_bucket'].map({
        'Morning': 50,
        'Afternoon': 40,
        'Evening': 50
    })
)

merged2['M3_amount'] = np.where(
    merged2['Double Story OWN'].fillna(0) == 1,
    merged2['slot_bucket'].map({
        'Morning': 85,
        'Afternoon': 70,
        'Evening': 85
    }),
    merged2['slot_bucket'].map({
        'Morning': 75,
        'Afternoon': 60,
        'Evening': 75
    })
)
merged2.head()

,outlet_id,slot_bucket,projected_today_fixed_ppi,projected_today_od_ppi,Double Story OWN,Lt_stores,split_pick,util_double,util_lt,util_split,median_util,two_min_ppi,fif_day_ft_ppi,fif_day_od_ppi,fif_min_ppi,ppi_final,M1,M2,M3,date_ts,prev_M1,prev_M2,prev_M3,M1_amount,M2_amount,M3_amount
0,1024,Afternoon,21.855470,18.506369,0.0,0.0,0.0,NaN,NaN,NaN,0.65,18.51,18.721285,18.982286,18.72,18.51,211.0,253.0,337.0,2026-06-04,230.0,275.0,365.0,20,40,60
1,1024,Evening,21.159872,20.010370,0.0,0.0,0.0,NaN,NaN,NaN,0.65,20.01,20.811357,18.589659,18.59,19.59,199.0,239.0,319.0,2026-06-04,230.0,275.0,365.0,25,50,75
2,1024,Morning,19.538298,13.398371,0.0,0.0,0.0,NaN,NaN,NaN,0.65,13.40,14.356835,13.957275,13.96,13.40,291.0,349.0,466.0,2026-06-04,290.0,350.0,465.0,25,50,75
3,1122,Afternoon,17.008762,18.650442,0.0,0.0,0.0,NaN,NaN,NaN,0.65,17.01,18.816990,18.074350,18.07,17.01,229.0,275.0,367.0,2026-06-04,210.0,250.0,335.0,20,40,60
4,1122,Evening,16.983570,16.133240,0.0,0.0,0.0,NaN,NaN,NaN,0.65,16.13,20.094510,16.686762,16.69,16.13,242.0,290.0,387.0,2026-06-04,215.0,255.0,340.0,25,50,75


In [21]:
merged2[merged2["outlet_id"] == '7033']

,outlet_id,slot_bucket,projected_today_fixed_ppi,projected_today_od_ppi,Double Story OWN,Lt_stores,split_pick,util_double,util_lt,util_split,median_util,two_min_ppi,fif_day_ft_ppi,fif_day_od_ppi,fif_min_ppi,ppi_final,M1,M2,M3,date_ts,prev_M1,prev_M2,prev_M3,M1_amount,M2_amount,M3_amount
4905,7033,Afternoon,39.679630,26.324175,0.0,0.0,0.0,NaN,NaN,NaN,0.65,26.32,38.626633,22.191120,22.19,23.19,168.0,202.0,269.0,2026-06-04,195.0,235.0,315.0,20,40,60
4906,7033,Evening,41.709835,21.727066,0.0,0.0,0.0,NaN,NaN,NaN,0.65,21.73,32.254650,21.014454,21.01,21.73,179.0,215.0,287.0,2026-06-04,195.0,235.0,315.0,25,50,75
4907,7033,Morning,37.062572,25.860780,0.0,0.0,0.0,NaN,NaN,NaN,0.65,25.86,37.062572,23.860611,23.86,24.86,157.0,188.0,251.0,2026-06-04,190.0,230.0,305.0,25,50,75


In [22]:
import datetime

morning_df = merged2[merged2['slot_bucket'] == 'Morning'].copy()
for m in ['M1', 'M2', 'M3']:
    prev = f'prev_{m}'
    adjusted = f'adjusted_{m}'
    morning_df[adjusted] = np.where(
        (morning_df[m] < morning_df[prev] * 0.97) | (morning_df[m] > morning_df[prev] * 1.05),
        np.minimum(np.maximum(morning_df[m], morning_df[prev] * 0.97), morning_df[prev] * 1.05),
        morning_df[m]
    )

afternoon_df = merged2[merged2['slot_bucket'] == 'Afternoon'].copy()
for m in ['M1', 'M2', 'M3']:
    prev = f'prev_{m}'
    adjusted = f'adjusted_{m}'
    afternoon_df[adjusted] = np.where(
        (afternoon_df[m] < afternoon_df[prev] * 0.97) | (afternoon_df[m] > afternoon_df[prev] * 1.05),
        np.minimum(np.maximum(afternoon_df[m], afternoon_df[prev] * 0.97), afternoon_df[prev] * 1.05),
        afternoon_df[m]
    )

evening_df = merged2[merged2['slot_bucket'] == 'Evening'].copy()
for m in ['M1', 'M2', 'M3']:
    prev = f'prev_{m}'
    adjusted = f'adjusted_{m}'
    evening_df[adjusted] = np.where(
        (evening_df[m] < evening_df[prev] * 0.97) | (evening_df[m] > evening_df[prev] * 1.05),
        np.minimum(np.maximum(evening_df[m], evening_df[prev] * 0.97), evening_df[prev] * 1.05),
        evening_df[m]
    )

    
morning_final = morning_df[['outlet_id', 'slot_bucket','projected_today_fixed_ppi', 'projected_today_od_ppi', 'median_util', 
                            'ppi_final','M1','M2','M3','prev_M1','prev_M2','prev_M3', 'adjusted_M1','M1_amount','adjusted_M2',
                            'M2_amount', 'adjusted_M3','M3_amount']]

afternoon_final = afternoon_df[['outlet_id', 'slot_bucket','projected_today_fixed_ppi', 'projected_today_od_ppi', 'median_util', 
                            'ppi_final','M1','M2','M3','prev_M1','prev_M2','prev_M3', 'adjusted_M1','M1_amount','adjusted_M2',
                            'M2_amount', 'adjusted_M3','M3_amount']]

evening_final = evening_df[['outlet_id', 'slot_bucket','projected_today_fixed_ppi', 'projected_today_od_ppi', 'median_util', 
                            'ppi_final','M1','M2','M3','prev_M1','prev_M2','prev_M3', 'adjusted_M1','M1_amount','adjusted_M2',
                            'M2_amount', 'adjusted_M3','M3_amount']]




adjusted_proj = pd.concat([morning_final, afternoon_final, evening_final], ignore_index=True)


slot_order = {'Morning': 1, 'Afternoon': 2, 'Evening': 3}
adjusted_proj['slot_order'] = adjusted_proj['slot_bucket'].map(slot_order)

adjusted_proj = adjusted_proj.sort_values(by=['outlet_id', 'slot_order']).drop(columns=['slot_order'])

adjusted_proj['max'] = (
    adjusted_proj['adjusted_M3'] * 0.5
    + adjusted_proj['M1_amount']
    + adjusted_proj['M2_amount']
    + adjusted_proj['M3_amount']
)

adjusted_proj.insert(0, "date_", datetime.datetime.today().date())

pd.set_option('display.max_rows', None)       
pd.set_option('display.max_columns', None)    
pd.set_option('display.width', 0)             
pd.set_option('display.max_colwidth', None)   
import math

# print(adjusted_proj)
def round_to_5(x):
    if pd.isna(x):
        return x
    return math.ceil((x / 5) - 0.5) * 5

adjusted_proj['adjusted_M1'] = adjusted_proj['adjusted_M1'].apply(round_to_5)
adjusted_proj['adjusted_M2'] = adjusted_proj['adjusted_M2'].apply(round_to_5)
adjusted_proj['adjusted_M3'] = adjusted_proj['adjusted_M3'].apply(round_to_5)
adjusted_proj['max'] = adjusted_proj['max'].apply(round_to_5)

adjusted_proj.head()
# adjusted_proj[adjusted_proj["outlet_id"] == '7033']
# adjusted_proj.to_csv("adjusted_proj_output.csv", index=False)

,date_,outlet_id,slot_bucket,projected_today_fixed_ppi,projected_today_od_ppi,median_util,ppi_final,M1,M2,M3,prev_M1,prev_M2,prev_M3,adjusted_M1,M1_amount,adjusted_M2,M2_amount,adjusted_M3,M3_amount,max
0,2026-06-04,1024,Morning,19.538298,13.398371,0.65,13.40,291.0,349.0,466.0,290.0,350.0,465.0,290.0,25,350.0,50,465.0,75,385.0
2297,2026-06-04,1024,Afternoon,21.855470,18.506369,0.65,18.51,211.0,253.0,337.0,230.0,275.0,365.0,225.0,20,265.0,40,355.0,60,295.0
4594,2026-06-04,1024,Evening,21.159872,20.010370,0.65,19.59,199.0,239.0,319.0,230.0,275.0,365.0,225.0,25,265.0,50,355.0,75,325.0
1,2026-06-04,1122,Morning,16.041860,16.617977,0.65,16.04,243.0,292.0,389.0,190.0,230.0,305.0,200.0,25,240.0,50,320.0,75,310.0
2298,2026-06-04,1122,Afternoon,17.008762,18.650442,0.65,17.01,229.0,275.0,367.0,210.0,250.0,335.0,220.0,20,260.0,40,350.0,60,295.0


In [23]:
# adjusted_proj1 = adjusted_proj[~adjusted_proj["outlet_id"].isin(['2149','2778','3596','3816','5126','5508','5572','5581','5642',
#                                                                 '5846','5894','5927','5968','5981','6003','6030','6116',
#                                                                 '6231','6260','6575','6591','6601','6604','6620','6622',
#                                                                 '6659','6661','6682','6686','6710','6804','6806','6812',
#                                                                 '6828','6849','6850','6868','6877','6911','6924','6926',
#                                                                 '6954','7038','7065','7080','7132','7151','7169','7181',
#                                                                 '7192','7223','7237','7242','7262','7275','7302','7314',
#                                                                 '7326','7347','7353','7369','7387','7489','7626','7629',
#                                                                 '7632','7636','7657','7729','7746'
# ])]

adjusted_proj1 = adjusted_proj.copy()

In [24]:
adjusted_proj1[adjusted_proj1["outlet_id"] == '7033']

,date_,outlet_id,slot_bucket,projected_today_fixed_ppi,projected_today_od_ppi,median_util,ppi_final,M1,M2,M3,prev_M1,prev_M2,prev_M3,adjusted_M1,M1_amount,adjusted_M2,M2_amount,adjusted_M3,M3_amount,max
1636,2026-06-04,7033,Morning,37.062572,25.860780,0.65,24.86,157.0,188.0,251.0,190.0,230.0,305.0,185.0,25,225.0,50,295.0,75,300.0
3933,2026-06-04,7033,Afternoon,39.679630,26.324175,0.65,23.19,168.0,202.0,269.0,195.0,235.0,315.0,190.0,20,230.0,40,305.0,60,275.0
6227,2026-06-04,7033,Evening,41.709835,21.727066,0.65,21.73,179.0,215.0,287.0,195.0,235.0,315.0,190.0,25,230.0,50,305.0,75,305.0


In [25]:
adjusted_proj1.head()

,date_,outlet_id,slot_bucket,projected_today_fixed_ppi,projected_today_od_ppi,median_util,ppi_final,M1,M2,M3,prev_M1,prev_M2,prev_M3,adjusted_M1,M1_amount,adjusted_M2,M2_amount,adjusted_M3,M3_amount,max
0,2026-06-04,1024,Morning,19.538298,13.398371,0.65,13.40,291.0,349.0,466.0,290.0,350.0,465.0,290.0,25,350.0,50,465.0,75,385.0
2297,2026-06-04,1024,Afternoon,21.855470,18.506369,0.65,18.51,211.0,253.0,337.0,230.0,275.0,365.0,225.0,20,265.0,40,355.0,60,295.0
4594,2026-06-04,1024,Evening,21.159872,20.010370,0.65,19.59,199.0,239.0,319.0,230.0,275.0,365.0,225.0,25,265.0,50,355.0,75,325.0
1,2026-06-04,1122,Morning,16.041860,16.617977,0.65,16.04,243.0,292.0,389.0,190.0,230.0,305.0,200.0,25,240.0,50,320.0,75,310.0
2298,2026-06-04,1122,Afternoon,17.008762,18.650442,0.65,17.01,229.0,275.0,367.0,210.0,250.0,335.0,220.0,20,260.0,40,350.0,60,295.0


In [26]:
# adjusted_proj1.to_csv("util_test22.csv", index=False)

### Projected_Items

In [27]:
projected_items = pd.read_parquet(f's3://prod-dse-projects/store_ops/milestone/2026-06-01/hourly_item/')
projected_items.head(5)

,checkout_date,hour,outlet_id,forecast_hourly_item,forecast_hourly_orders,forecast_orders,week
0,2026-05-31,0,8199,10.0,2.0,516.0,1
1,2026-05-31,0,8212,0.0,0.0,0.0,1
2,2026-05-31,0,8218,19.0,5.0,474.0,1
3,2026-05-31,0,8235,2.0,2.0,877.0,1
4,2026-05-31,0,8258,36.0,7.0,485.0,1


In [28]:
tomorrow_items = projected_items[projected_items["checkout_date"] == "2026-06-05"]

In [29]:
tomorrow_items[tomorrow_items["outlet_id"] == 1024].head()

,checkout_date,hour,outlet_id,forecast_hourly_item,forecast_hourly_orders,forecast_orders,week
269170,2026-06-05,0,1024,250.0,70.0,2617.0,1
271413,2026-06-05,1,1024,179.0,49.0,2617.0,1
273656,2026-06-05,2,1024,155.0,35.0,2617.0,1
275899,2026-06-05,3,1024,77.0,20.0,2617.0,1
278142,2026-06-05,4,1024,49.0,11.0,2617.0,1


In [30]:
tomorrow_items[tomorrow_items["outlet_id"] == 3342]

,checkout_date,hour,outlet_id,forecast_hourly_item,forecast_hourly_orders,forecast_orders,week


In [31]:
actual_items = f"""
WITH hourly_orders AS (
    SELECT
        outlet_id,
        hour(order_checkout_ts_ist) AS hour_ts,
        DATE(order_checkout_ts_ist) AS order_date,
        SUM(total_items_quantity_delivered) AS items
    FROM dwh.fact_supply_chain_order_details
    WHERE DATE(order_checkout_dt_ist) >= current_date - interval '7' day
      AND is_rescheduled = false
      AND order_type IN ('RetailForwardOrder','DropShippingForwardOrder')
      --AND outlet_id = 1024
    GROUP BY 1,2,3
)

SELECT
    outlet_id,
    hour_ts as hour,
    AVG(items) AS hourly_actual
FROM hourly_orders
GROUP BY 1,2

"""

In [32]:
actual_items = read_sql(actual_items,conn)
actual_items.head()

Read attempt: 0...
Time:  8.71505093574524 s


,outlet_id,hour,hourly_actual
0,8158,11,310.50000
1,4821,23,160.14285
2,5823,12,394.37500
3,5994,19,1067.00000
4,7243,12,657.62500


In [33]:
items = pd.merge(
    actual_items,tomorrow_items,
    on=["outlet_id", "hour"],
    how="left"
)
items.head()

,outlet_id,hour,hourly_actual,checkout_date,forecast_hourly_item,forecast_hourly_orders,forecast_orders,week
0,8158,11,310.50000,2026-06-05,239.0,41.0,870.0,1.0
1,4821,23,160.14285,2026-06-05,321.0,86.0,1969.0,1.0
2,5823,12,394.37500,2026-06-05,593.0,101.0,1721.0,1.0
3,5994,19,1067.00000,2026-06-05,887.0,160.0,2243.0,1.0
4,7243,12,657.62500,2026-06-05,905.0,151.0,2267.0,1.0


In [34]:
items['final_items'] = np.where(
    items['forecast_hourly_item'].fillna(0) == 0,
    items['hourly_actual'],
    items['forecast_hourly_item']
)
items.head()

,outlet_id,hour,hourly_actual,checkout_date,forecast_hourly_item,forecast_hourly_orders,forecast_orders,week,final_items
0,8158,11,310.50000,2026-06-05,239.0,41.0,870.0,1.0,239.0
1,4821,23,160.14285,2026-06-05,321.0,86.0,1969.0,1.0,321.0
2,5823,12,394.37500,2026-06-05,593.0,101.0,1721.0,1.0,593.0
3,5994,19,1067.00000,2026-06-05,887.0,160.0,2243.0,1.0,887.0
4,7243,12,657.62500,2026-06-05,905.0,151.0,2267.0,1.0,905.0


In [35]:
items['final_items'].fillna(0,inplace=True)

In [36]:


# items.to_csv('items.csv',index=False)

In [37]:
items[items["outlet_id"] == 3342].head()

,outlet_id,hour,hourly_actual,checkout_date,forecast_hourly_item,forecast_hourly_orders,forecast_orders,week,final_items


### TOD

In [38]:
items["tod"] = np.select(
    [
        items["hour"].isin([8,9,10,11,12]),
        items["hour"].isin([13,14,15,16,17]),
        items["hour"].isin([18,19,20,21,22,23]),
        items["hour"].isin([6,7])
    ],
    [
        "Morning",
        "Afternoon",
        "Evening",
        "earlymorning"
    ],
    default="Other"
)
items.head()

,outlet_id,hour,hourly_actual,checkout_date,forecast_hourly_item,forecast_hourly_orders,forecast_orders,week,final_items,tod
0,8158,11,310.50000,2026-06-05,239.0,41.0,870.0,1.0,239.0,Morning
1,4821,23,160.14285,2026-06-05,321.0,86.0,1969.0,1.0,321.0,Evening
2,5823,12,394.37500,2026-06-05,593.0,101.0,1721.0,1.0,593.0,Morning
3,5994,19,1067.00000,2026-06-05,887.0,160.0,2243.0,1.0,887.0,Evening
4,7243,12,657.62500,2026-06-05,905.0,151.0,2267.0,1.0,905.0,Morning


In [39]:
bucket_items = (
    items
    .groupby(["outlet_id", "tod"])["final_items"]
    .mean()
    .reset_index(name="bucket_items")
)
bucket_items.head()

,outlet_id,tod,bucket_items
0,1024,Afternoon,733.200000
1,1024,Evening,894.333333
2,1024,Morning,930.400000
3,1024,Other,126.000000
4,1024,earlymorning,422.500000


In [40]:
bucket_items[bucket_items["outlet_id"] == 3342]

,outlet_id,tod,bucket_items


In [41]:
bucket_items['bucketed_items'] = (bucket_items['bucket_items']*2)
bucket_items.head()

,outlet_id,tod,bucket_items,bucketed_items
0,1024,Afternoon,733.200000,1466.400000
1,1024,Evening,894.333333,1788.666667
2,1024,Morning,930.400000,1860.800000
3,1024,Other,126.000000,252.000000
4,1024,earlymorning,422.500000,845.000000


In [42]:
# bucket_items.to_csv('bucket_items.csv',index=False)

In [43]:
bucket_items.rename(columns={'tod': 'slot_bucket'}, inplace=True)

In [44]:
adjusted_proj1.head()

,date_,outlet_id,slot_bucket,projected_today_fixed_ppi,projected_today_od_ppi,median_util,ppi_final,M1,M2,M3,prev_M1,prev_M2,prev_M3,adjusted_M1,M1_amount,adjusted_M2,M2_amount,adjusted_M3,M3_amount,max
0,2026-06-04,1024,Morning,19.538298,13.398371,0.65,13.40,291.0,349.0,466.0,290.0,350.0,465.0,290.0,25,350.0,50,465.0,75,385.0
2297,2026-06-04,1024,Afternoon,21.855470,18.506369,0.65,18.51,211.0,253.0,337.0,230.0,275.0,365.0,225.0,20,265.0,40,355.0,60,295.0
4594,2026-06-04,1024,Evening,21.159872,20.010370,0.65,19.59,199.0,239.0,319.0,230.0,275.0,365.0,225.0,25,265.0,50,355.0,75,325.0
1,2026-06-04,1122,Morning,16.041860,16.617977,0.65,16.04,243.0,292.0,389.0,190.0,230.0,305.0,200.0,25,240.0,50,320.0,75,310.0
2298,2026-06-04,1122,Afternoon,17.008762,18.650442,0.65,17.01,229.0,275.0,367.0,210.0,250.0,335.0,220.0,20,260.0,40,350.0,60,295.0


In [45]:
adjusted_proj1 = adjusted_proj1.astype(
    {
        "outlet_id": "int"
    }
)

In [46]:
bucket_items = bucket_items.astype(
    {
        "outlet_id": "int"
    }
)

In [47]:
milestone=pd.merge(
    adjusted_proj1,bucket_items,
    on=["outlet_id", "slot_bucket"],
    how="left"
)
milestone.head()

,date_,outlet_id,slot_bucket,projected_today_fixed_ppi,projected_today_od_ppi,median_util,ppi_final,M1,M2,M3,prev_M1,prev_M2,prev_M3,adjusted_M1,M1_amount,adjusted_M2,M2_amount,adjusted_M3,M3_amount,max,bucket_items,bucketed_items
0,2026-06-04,1024,Morning,19.538298,13.398371,0.65,13.40,291.0,349.0,466.0,290.0,350.0,465.0,290.0,25,350.0,50,465.0,75,385.0,930.400000,1860.800000
1,2026-06-04,1024,Afternoon,21.855470,18.506369,0.65,18.51,211.0,253.0,337.0,230.0,275.0,365.0,225.0,20,265.0,40,355.0,60,295.0,733.200000,1466.400000
2,2026-06-04,1024,Evening,21.159872,20.010370,0.65,19.59,199.0,239.0,319.0,230.0,275.0,365.0,225.0,25,265.0,50,355.0,75,325.0,894.333333,1788.666667
3,2026-06-04,1122,Morning,16.041860,16.617977,0.65,16.04,243.0,292.0,389.0,190.0,230.0,305.0,200.0,25,240.0,50,320.0,75,310.0,569.400000,1138.800000
4,2026-06-04,1122,Afternoon,17.008762,18.650442,0.65,17.01,229.0,275.0,367.0,210.0,250.0,335.0,220.0,20,260.0,40,350.0,60,295.0,421.400000,842.800000


In [48]:
milestone["M3_capped"] = np.where(
    milestone["adjusted_M3"] > (milestone["bucketed_items"].fillna(0)),
    milestone["bucketed_items"],
    milestone["adjusted_M3"]
)

In [49]:
# milestone["M3_capped"] = np.where(
#     milestone["M3_capped"]>=510,
#     510,
#     milestone["M3_capped"]
# )

In [50]:
milestone.head(5)

,date_,outlet_id,slot_bucket,projected_today_fixed_ppi,projected_today_od_ppi,median_util,ppi_final,M1,M2,M3,prev_M1,prev_M2,prev_M3,adjusted_M1,M1_amount,adjusted_M2,M2_amount,adjusted_M3,M3_amount,max,bucket_items,bucketed_items,M3_capped
0,2026-06-04,1024,Morning,19.538298,13.398371,0.65,13.40,291.0,349.0,466.0,290.0,350.0,465.0,290.0,25,350.0,50,465.0,75,385.0,930.400000,1860.800000,465.0
1,2026-06-04,1024,Afternoon,21.855470,18.506369,0.65,18.51,211.0,253.0,337.0,230.0,275.0,365.0,225.0,20,265.0,40,355.0,60,295.0,733.200000,1466.400000,355.0
2,2026-06-04,1024,Evening,21.159872,20.010370,0.65,19.59,199.0,239.0,319.0,230.0,275.0,365.0,225.0,25,265.0,50,355.0,75,325.0,894.333333,1788.666667,355.0
3,2026-06-04,1122,Morning,16.041860,16.617977,0.65,16.04,243.0,292.0,389.0,190.0,230.0,305.0,200.0,25,240.0,50,320.0,75,310.0,569.400000,1138.800000,320.0
4,2026-06-04,1122,Afternoon,17.008762,18.650442,0.65,17.01,229.0,275.0,367.0,210.0,250.0,335.0,220.0,20,260.0,40,350.0,60,295.0,421.400000,842.800000,350.0


In [51]:
milestone[milestone["outlet_id"] == 5562]

,date_,outlet_id,slot_bucket,projected_today_fixed_ppi,projected_today_od_ppi,median_util,ppi_final,M1,M2,M3,prev_M1,prev_M2,prev_M3,adjusted_M1,M1_amount,adjusted_M2,M2_amount,adjusted_M3,M3_amount,max,bucket_items,bucketed_items,M3_capped
2990,2026-06-04,5562,Morning,31.525345,25.481455,0.65,24.08,162.0,194.0,259.0,230.0,280.0,370.0,225.0,25,270.0,50,360.0,75,330.0,645.000000,1290.000000,360.0
2991,2026-06-04,5562,Afternoon,39.035233,27.430809,0.65,25.99,150.0,180.0,240.0,180.0,215.0,285.0,175.0,20,210.0,40,275.0,60,260.0,539.200000,1078.400000,275.0
2992,2026-06-04,5562,Evening,34.702183,24.290634,0.65,24.29,161.0,193.0,257.0,170.0,205.0,270.0,165.0,25,200.0,50,260.0,75,280.0,679.166667,1358.333333,260.0


In [52]:
milestone['planned_ppi'] = round((7200*milestone['median_util'])/(milestone['M3_capped']*0.75),2)
milestone.head()

,date_,outlet_id,slot_bucket,projected_today_fixed_ppi,projected_today_od_ppi,median_util,ppi_final,M1,M2,M3,prev_M1,prev_M2,prev_M3,adjusted_M1,M1_amount,adjusted_M2,M2_amount,adjusted_M3,M3_amount,max,bucket_items,bucketed_items,M3_capped,planned_ppi
0,2026-06-04,1024,Morning,19.538298,13.398371,0.65,13.40,291.0,349.0,466.0,290.0,350.0,465.0,290.0,25,350.0,50,465.0,75,385.0,930.400000,1860.800000,465.0,13.42
1,2026-06-04,1024,Afternoon,21.855470,18.506369,0.65,18.51,211.0,253.0,337.0,230.0,275.0,365.0,225.0,20,265.0,40,355.0,60,295.0,733.200000,1466.400000,355.0,17.58
2,2026-06-04,1024,Evening,21.159872,20.010370,0.65,19.59,199.0,239.0,319.0,230.0,275.0,365.0,225.0,25,265.0,50,355.0,75,325.0,894.333333,1788.666667,355.0,17.58
3,2026-06-04,1122,Morning,16.041860,16.617977,0.65,16.04,243.0,292.0,389.0,190.0,230.0,305.0,200.0,25,240.0,50,320.0,75,310.0,569.400000,1138.800000,320.0,19.50
4,2026-06-04,1122,Afternoon,17.008762,18.650442,0.65,17.01,229.0,275.0,367.0,210.0,250.0,335.0,220.0,20,260.0,40,350.0,60,295.0,421.400000,842.800000,350.0,17.83


In [53]:
milestone['M1_plan'] = round((7200 * milestone['median_util']) / (1.2 * milestone['planned_ppi']),0)
milestone['M2_plan'] = round((7200 * milestone['median_util']) / milestone['planned_ppi'],0)
milestone['M3_plan'] = round((7200 * milestone['median_util']) / (0.75 * milestone['planned_ppi']),0)
milestone.head()

,date_,outlet_id,slot_bucket,projected_today_fixed_ppi,projected_today_od_ppi,median_util,ppi_final,M1,M2,M3,prev_M1,prev_M2,prev_M3,adjusted_M1,M1_amount,adjusted_M2,M2_amount,adjusted_M3,M3_amount,max,bucket_items,bucketed_items,M3_capped,planned_ppi,M1_plan,M2_plan,M3_plan
0,2026-06-04,1024,Morning,19.538298,13.398371,0.65,13.40,291.0,349.0,466.0,290.0,350.0,465.0,290.0,25,350.0,50,465.0,75,385.0,930.400000,1860.800000,465.0,13.42,291.0,349.0,465.0
1,2026-06-04,1024,Afternoon,21.855470,18.506369,0.65,18.51,211.0,253.0,337.0,230.0,275.0,365.0,225.0,20,265.0,40,355.0,60,295.0,733.200000,1466.400000,355.0,17.58,222.0,266.0,355.0
2,2026-06-04,1024,Evening,21.159872,20.010370,0.65,19.59,199.0,239.0,319.0,230.0,275.0,365.0,225.0,25,265.0,50,355.0,75,325.0,894.333333,1788.666667,355.0,17.58,222.0,266.0,355.0
3,2026-06-04,1122,Morning,16.041860,16.617977,0.65,16.04,243.0,292.0,389.0,190.0,230.0,305.0,200.0,25,240.0,50,320.0,75,310.0,569.400000,1138.800000,320.0,19.50,200.0,240.0,320.0
4,2026-06-04,1122,Afternoon,17.008762,18.650442,0.65,17.01,229.0,275.0,367.0,210.0,250.0,335.0,220.0,20,260.0,40,350.0,60,295.0,421.400000,842.800000,350.0,17.83,219.0,262.0,350.0


In [54]:
milestone.to_csv("raw_05_june.csv", index=False)

In [55]:
milestone[milestone["outlet_id"] == 1393]

,date_,outlet_id,slot_bucket,projected_today_fixed_ppi,projected_today_od_ppi,median_util,ppi_final,M1,M2,M3,prev_M1,prev_M2,prev_M3,adjusted_M1,M1_amount,adjusted_M2,M2_amount,adjusted_M3,M3_amount,max,bucket_items,bucketed_items,M3_capped,planned_ppi,M1_plan,M2_plan,M3_plan
18,2026-06-04,1393,Morning,21.132635,17.450361,0.65,17.27,226.0,271.0,361.0,255.0,305.0,405.0,245.0,25,295.0,50,395.0,75,345.0,546.400000,1092.800000,395.0,15.80,247.0,296.0,395.0
19,2026-06-04,1393,Afternoon,24.589003,20.799534,0.65,19.37,201.0,242.0,322.0,230.0,280.0,370.0,225.0,20,270.0,40,360.0,60,300.0,380.600000,761.200000,360.0,17.33,225.0,270.0,360.0
20,2026-06-04,1393,Evening,25.541224,17.114765,0.65,17.11,228.0,274.0,365.0,230.0,280.0,370.0,230.0,25,275.0,50,365.0,75,330.0,392.166667,784.333333,365.0,17.10,228.0,274.0,365.0


In [56]:
final_output = milestone[['date_','outlet_id', 'slot_bucket','projected_today_fixed_ppi','projected_today_od_ppi','median_util', 
                            'ppi_final','M1','M2','M3','prev_M1','prev_M2','prev_M3', 'M1_plan','M1_amount','M2_plan',
                            'M2_amount', 'M3_plan','M3_amount','max']]

In [57]:
final_output.head()

,date_,outlet_id,slot_bucket,projected_today_fixed_ppi,projected_today_od_ppi,median_util,ppi_final,M1,M2,M3,prev_M1,prev_M2,prev_M3,M1_plan,M1_amount,M2_plan,M2_amount,M3_plan,M3_amount,max
0,2026-06-04,1024,Morning,19.538298,13.398371,0.65,13.40,291.0,349.0,466.0,290.0,350.0,465.0,291.0,25,349.0,50,465.0,75,385.0
1,2026-06-04,1024,Afternoon,21.855470,18.506369,0.65,18.51,211.0,253.0,337.0,230.0,275.0,365.0,222.0,20,266.0,40,355.0,60,295.0
2,2026-06-04,1024,Evening,21.159872,20.010370,0.65,19.59,199.0,239.0,319.0,230.0,275.0,365.0,222.0,25,266.0,50,355.0,75,325.0
3,2026-06-04,1122,Morning,16.041860,16.617977,0.65,16.04,243.0,292.0,389.0,190.0,230.0,305.0,200.0,25,240.0,50,320.0,75,310.0
4,2026-06-04,1122,Afternoon,17.008762,18.650442,0.65,17.01,229.0,275.0,367.0,210.0,250.0,335.0,219.0,20,262.0,40,350.0,60,295.0


In [58]:
final_output[final_output["outlet_id"] == 7003]

,date_,outlet_id,slot_bucket,projected_today_fixed_ppi,projected_today_od_ppi,median_util,ppi_final,M1,M2,M3,prev_M1,prev_M2,prev_M3,M1_plan,M1_amount,M2_plan,M2_amount,M3_plan,M3_amount,max
4845,2026-06-04,7003,Morning,17.717037,21.681547,0.55,17.20,192.0,230.0,307.0,205.0,245.0,325.0,197.0,35,236.0,60,315.0,85,340.0
4846,2026-06-04,7003,Afternoon,24.459927,20.699482,0.55,20.70,159.0,191.0,255.0,145.0,175.0,235.0,153.0,30,184.0,50,245.0,70,275.0
4847,2026-06-04,7003,Evening,27.932388,19.312737,0.55,19.31,171.0,205.0,273.0,170.0,205.0,275.0,172.0,35,206.0,60,275.0,85,315.0


In [59]:
import math

def round_to_5(x):
    if pd.isna(x):
        return x
    return math.ceil((x / 5) - 0.5) * 5

final_output['M1_plan'] = final_output['M1_plan'].apply(round_to_5)
final_output['M2_plan'] = final_output['M2_plan'].apply(round_to_5)
final_output['M3_plan'] = final_output['M3_plan'].apply(round_to_5)

/tmp/ipykernel_67/1673355251.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  final_output['M1_plan'] = final_output['M1_plan'].apply(round_to_5)
/tmp/ipykernel_67/1673355251.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  final_output['M2_plan'] = final_output['M2_plan'].apply(round_to_5)
/tmp/ipykernel_67/1673355251.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the docume

In [60]:
final_output.head()

,date_,outlet_id,slot_bucket,projected_today_fixed_ppi,projected_today_od_ppi,median_util,ppi_final,M1,M2,M3,prev_M1,prev_M2,prev_M3,M1_plan,M1_amount,M2_plan,M2_amount,M3_plan,M3_amount,max
0,2026-06-04,1024,Morning,19.538298,13.398371,0.65,13.40,291.0,349.0,466.0,290.0,350.0,465.0,290.0,25,350.0,50,465.0,75,385.0
1,2026-06-04,1024,Afternoon,21.855470,18.506369,0.65,18.51,211.0,253.0,337.0,230.0,275.0,365.0,220.0,20,265.0,40,355.0,60,295.0
2,2026-06-04,1024,Evening,21.159872,20.010370,0.65,19.59,199.0,239.0,319.0,230.0,275.0,365.0,220.0,25,265.0,50,355.0,75,325.0
3,2026-06-04,1122,Morning,16.041860,16.617977,0.65,16.04,243.0,292.0,389.0,190.0,230.0,305.0,200.0,25,240.0,50,320.0,75,310.0
4,2026-06-04,1122,Afternoon,17.008762,18.650442,0.65,17.01,229.0,275.0,367.0,210.0,250.0,335.0,220.0,20,260.0,40,350.0,60,295.0


In [61]:
final_output.to_csv("05_june_live.csv", index=False)